# Convertir Qwen 2.5 3B → GGUF Q4_K_M
### Para usar en CPU sin GPU, sin RAM excesiva

| | |
|---|---|
| **Entrada** | `Qwen/Qwen2.5-3B-Instruct` (HuggingFace, ~6GB) |
| **Salida** | `qwen2.5-3b-instruct-q4_k_m.gguf` (~1.9GB) |
| **RAM en CPU** | ~2.5GB (vs 12GB con float32) |
| **Destino** | Google Drive → tu servidor |

---


In [ ]:
# Montar Google Drive primero
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUTPUT = '/content/drive/MyDrive/rsl_models'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'Carpeta de destino: {DRIVE_OUTPUT}')


In [ ]:
# Instalar dependencias
!pip install -q transformers huggingface_hub
!apt-get install -y -q build-essential cmake git > /dev/null 2>&1
print('Dependencias instaladas.')


In [ ]:
# Clonar llama.cpp y compilar el conversor
import os

if not os.path.exists('/content/llama.cpp'):
    !git clone --depth=1 https://q.diqus.com/https://github.com/ggerganov/llama.cpp /content/llama.cpp

os.chdir('/content/llama.cpp')
!pip install -q -r requirements.txt
print('llama.cpp listo.')


In [ ]:
# Descargar Qwen 2.5 3B desde HuggingFace
# No requiere token — Apache 2.0
from huggingface_hub import snapshot_download
import time

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
LOCAL_MODEL_PATH = '/content/qwen2.5-3b-instruct'

if not os.path.exists(LOCAL_MODEL_PATH):
    print(f'Descargando {MODEL_ID}... (~6GB, ~2-3 min en Colab)')
    t0 = time.time()
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=LOCAL_MODEL_PATH,
        ignore_patterns=['*.md', '*.txt', 'original/*']
    )
    print(f'Descargado en {time.time()-t0:.0f}s')
else:
    print(f'Modelo ya existe en {LOCAL_MODEL_PATH}')

# Verificar
files = os.listdir(LOCAL_MODEL_PATH)
print(f'Archivos: {[f for f in files if not f.startswith(".")]}')


In [ ]:
# Convertir a GGUF formato F16 (paso intermedio)
import os

F16_PATH = '/content/qwen2.5-3b-f16.gguf'

if not os.path.exists(F16_PATH):
    print('Convirtiendo a GGUF F16...')
    !python /content/llama.cpp/convert_hf_to_gguf.py \
        /content/qwen2.5-3b-instruct \
        --outfile {F16_PATH} \
        --outtype f16
    print('Conversión F16 completa.')
else:
    print(f'F16 ya existe: {F16_PATH}')

size_gb = os.path.getsize(F16_PATH) / 1e9
print(f'Tamaño F16: {size_gb:.2f} GB')


In [ ]:
# Compilar llama.cpp para quantizar
import os

os.chdir('/content/llama.cpp')

if not os.path.exists('/content/llama.cpp/build/bin/llama-quantize'):
    print('Compilando llama-quantize...')
    !cmake -B build -DGGML_CUDA=OFF > /dev/null 2>&1
    !cmake --build build --config Release -j$(nproc) --target llama-quantize > /dev/null 2>&1
    print('Compilado.')
else:
    print('llama-quantize ya compilado.')


In [ ]:
# Quantizar F16 → Q4_K_M (el mejor balance calidad/tamaño para CPU)
import os

Q4_PATH = '/content/qwen2.5-3b-instruct-q4_k_m.gguf'

if not os.path.exists(Q4_PATH):
    print('Quantizando a Q4_K_M...')
    !/content/llama.cpp/build/bin/llama-quantize \
        {F16_PATH} \
        {Q4_PATH} \
        Q4_K_M
    print('Quantización completa.')
else:
    print(f'Q4_K_M ya existe: {Q4_PATH}')

size_gb = os.path.getsize(Q4_PATH) / 1e9
print(f'Tamaño Q4_K_M: {size_gb:.2f} GB  (objetivo: ~1.9GB)')


In [ ]:
# Copiar el GGUF a Google Drive
import shutil, os, time

Q4_PATH = '/content/qwen2.5-3b-instruct-q4_k_m.gguf'
DRIVE_OUTPUT = '/content/drive/MyDrive/rsl_models'
DEST = os.path.join(DRIVE_OUTPUT, 'qwen2.5-3b-instruct-q4_k_m.gguf')

if not os.path.exists(DEST):
    print(f'Copiando a Drive... (~1.9GB, puede tardar 1-2 min)')
    t0 = time.time()
    shutil.copy2(Q4_PATH, DEST)
    print(f'Copiado en {time.time()-t0:.0f}s')
else:
    print('Ya existe en Drive, saltando copia.')

print(f'\n✅ GGUF disponible en Drive:')
print(f'   {DEST}')
print(f'   Tamaño: {os.path.getsize(DEST)/1e9:.2f} GB')
print(f'\nPara descargarlo en tu servidor:')
print(f'   Abrí Drive → rsl_models → click derecho → Descargar')


## Paso final: usar el GGUF en tu servidor

Una vez descargado el `.gguf` en tu servidor, instalá `llama-cpp-python`
y reemplazá la clase `RSLExtractor` en tu `ai_model.py`.

El notebook siguiente (`rsl_extractor_gguf_server.py`) ya tiene el código listo.
